In [1]:
# Monet CycleGAN (ResNet + Augmentation + Feature Matching + Multi-Scale Discriminator)

import os, random, zipfile
from glob import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision

print("TensorFlow:", tf.__version__)

tpu_available = False
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    strategy = tf.distribute.TPUStrategy(resolver)
    tpu_available = True
except ValueError:
    strategy = tf.distribute.get_strategy()

# configure
IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3
BUFFER_SIZE = 1024
BATCH_SIZE = 4
EPOCHS = 40
LAMBDA_CYCLE = 7.0
LAMBDA_ID = 0.1
NUM_GENERATED_IMAGES = 7000

MONET_TFREC_PATTERN = "/kaggle/input/gan-getting-started/monet_tfrec/*.tfrec"
PHOTO_TFREC_PATTERN = "/kaggle/input/gan-getting-started/photo_tfrec/*.tfrec"

OUTPUT_DIR = "/kaggle/working/monet_generated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# data
IMAGE_FEATURE_DESCRIPTION = {
    "image_name": tf.io.FixedLenFeature([], tf.string),
    "image":      tf.io.FixedLenFeature([], tf.string),
}

def parse_tfrecord(example):
    ex  = tf.io.parse_single_example(example, IMAGE_FEATURE_DESCRIPTION)
    img = tf.image.decode_jpeg(ex["image"], channels=IMG_CHANNELS)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = (tf.cast(img, tf.float32) / 127.5) - 1
    return img

def random_jitter(img):
    img = tf.image.resize(img, [286, 286])
    img = tf.image.random_crop(img, [IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.05)
    img = tf.image.random_contrast(img, 0.95, 1.05)
    img = tf.image.random_saturation(img, 0.95, 1.05)
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img

def load_dataset(pattern, batch, augment=False):
    files = tf.io.gfile.glob(pattern)
    ds    = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)
    ds    = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(lambda x: random_jitter(x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(BUFFER_SIZE).repeat().batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

# model comp
class InstanceNormalization(tf.keras.layers.Layer):
    def __init__(self, eps=1e-5, **kw):
        super().__init__(**kw); self.eps = eps
    def build(self, shape):
        c = shape[-1]
        self.gamma = self.add_weight(shape=(c,), initializer="ones",  trainable=True)
        self.beta  = self.add_weight(shape=(c,), initializer="zeros", trainable=True)
    def call(self, x):
        m, v = tf.nn.moments(x, [1, 2], keepdims=True)
        return self.gamma * (x - m) / tf.sqrt(v + self.eps) + self.beta

def resnet_block(x, f):
    init = tf.random_normal_initializer(0., 0.02)
    y = tf.keras.layers.Conv2D(f, 3, padding='same', kernel_initializer=init, use_bias=False)(x)
    y = InstanceNormalization()(y); y = tf.keras.layers.ReLU()(y)
    y = tf.keras.layers.Conv2D(f, 3, padding='same', kernel_initializer=init, use_bias=False)(y)
    y = InstanceNormalization()(y)
    return tf.keras.layers.Add()([x, y])

def build_resnet_generator(num_blocks=9):
    init = tf.random_normal_initializer(0., 0.02)
    inp  = tf.keras.layers.Input([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])

    x = tf.keras.layers.Conv2D(64, 7, padding='same', kernel_initializer=init, use_bias=False)(inp)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(128, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(256, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    for _ in range(num_blocks):
        x = resnet_block(x, 256)

    x = tf.keras.layers.Conv2DTranspose(128, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x   = tf.keras.layers.Conv2D(3, 7, padding='same', kernel_initializer=init)(x)
    out = tf.keras.layers.Activation('tanh')(x)
    return tf.keras.Model(inp, out, name="ResNetGen")

def build_discriminator_with_features(name="PatchDisc"):
    init = tf.random_normal_initializer(0., 0.02)
    inp  = tf.keras.layers.Input([None, None, IMG_CHANNELS])

    x1 = tf.keras.layers.Conv2D(64, 4, strides=2, padding='same', kernel_initializer=init)(inp)
    x1 = tf.keras.layers.LeakyReLU(0.2)(x1)

    x2 = tf.keras.layers.Conv2D(128, 4, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x1)
    x2 = InstanceNormalization()(x2); x2 = tf.keras.layers.LeakyReLU(0.2)(x2)

    x3 = tf.keras.layers.Conv2D(256, 4, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x2)
    x3 = InstanceNormalization()(x3); x3 = tf.keras.layers.LeakyReLU(0.2)(x3)

    x4 = tf.keras.layers.Conv2D(512, 4, padding='same', kernel_initializer=init, use_bias=False)(x3)
    x4 = InstanceNormalization()(x4); x4 = tf.keras.layers.LeakyReLU(0.2)(x4)

    out = tf.keras.layers.Conv2D(1, 4, padding='same', kernel_initializer=init)(x4)
    return tf.keras.Model(inp, [out, x1, x2, x3], name=name)

def build_multiscale_discriminator(name_prefix="MS"):
    inp        = tf.keras.layers.Input([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
    downscaled = tf.keras.layers.AveragePooling2D(pool_size=2)(inp)

    disc_full = build_discriminator_with_features(name=f"{name_prefix}_Full")
    disc_half = build_discriminator_with_features(name=f"{name_prefix}_Half")

    full_outs = disc_full(inp)
    half_outs = disc_half(downscaled)
    return tf.keras.Model(inp, full_outs + half_outs, name=f"{name_prefix}_MultiScale")

# loss calc
mse_loss = tf.keras.losses.MeanSquaredError()

def generator_gan_loss(fake):
    return mse_loss(tf.ones_like(fake), fake)

def discriminator_loss(real, fake):
    return 0.5*(mse_loss(tf.ones_like(real), real) + mse_loss(tf.zeros_like(fake), fake))

def cycle_loss(r, c):
    r = tf.cast(r, tf.float32); c = tf.cast(c, tf.float32)
    return LAMBDA_CYCLE * tf.reduce_mean(tf.abs(r - c))

def identity_loss(r, s):
    r = tf.cast(r, tf.float32); s = tf.cast(s, tf.float32)
    return LAMBDA_CYCLE * LAMBDA_ID * tf.reduce_mean(tf.abs(r - s))

def feature_matching_loss(real_feats, fake_feats, weight=10.0):
    loss = 0.0
    for r, f in zip(real_feats, fake_feats):
        loss += tf.reduce_mean(tf.abs(tf.cast(r, tf.float32) - tf.cast(f, tf.float32)))
    return loss * weight

def multiscale_disc_loss(disc, real, fake):
    r = disc(real, training=True); f = disc(fake, training=True)
    return 0.5*(discriminator_loss(r[0], f[0]) + discriminator_loss(r[4], f[4]))

def multiscale_gen_loss(disc, real, fake, fw=10.0):
    r = disc(real, training=False); f = disc(fake, training=True)
    adv = generator_gan_loss(f[0]) + generator_gan_loss(f[4])
    fm  = feature_matching_loss(r[1:4], f[1:4], fw) + feature_matching_loss(r[5:8], f[5:8], fw)
    return adv + fm

# replay buffer
class ImageBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size; self.buffer = []
    def query(self, images):
        out = []
        for img in tf.unstack(images):
            if len(self.buffer) < self.max_size:
                self.buffer.append(img); out.append(img)
            elif random.random() > 0.5:
                idx = random.randint(0, self.max_size - 1)
                out.append(self.buffer[idx]); self.buffer[idx] = img
            else:
                out.append(img)
        return tf.stack(out)

with strategy.scope():
    if tpu_available:
        mixed_precision.set_global_policy("mixed_bfloat16")

    generator_g     = build_resnet_generator()
    generator_f     = build_resnet_generator()
    discriminator_x = build_multiscale_discriminator("Dx")
    discriminator_y = build_multiscale_discriminator("Dy")

    lr = 2e-4
    g_g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    g_f_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    d_x_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    d_y_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)

fake_x_buffer = ImageBuffer()
fake_y_buffer = ImageBuffer()

# data
monet_ds = load_dataset(MONET_TFREC_PATTERN, BATCH_SIZE, augment=True)
photo_ds = load_dataset(PHOTO_TFREC_PATTERN, BATCH_SIZE, augment=True)
train_ds = tf.data.Dataset.zip((photo_ds, monet_ds))
train_ds = strategy.experimental_distribute_dataset(train_ds)

def update_lr(epoch):
    if epoch > EPOCHS // 2:
        new_lr = max(2e-4 * (1 - (epoch - EPOCHS//2) / (EPOCHS//2)), 1e-7)
        for opt in [g_g_opt, g_f_opt, d_x_opt, d_y_opt]:
            opt.learning_rate.assign(new_lr)

def train_step(real_x, real_y):
    with tf.GradientTape(persistent=True) as tape:
        fake_y   = generator_g(real_x, training=True)
        fake_x   = generator_f(real_y, training=True)
        cycled_x = generator_f(fake_y, training=True)
        cycled_y = generator_g(fake_x, training=True)
        same_x   = generator_f(real_x, training=True)
        same_y   = generator_g(real_y, training=True)

        fake_y_buf = fake_y_buffer.query(fake_y)
        fake_x_buf = fake_x_buffer.query(fake_x)

        cyc = cycle_loss(real_x, cycled_x) + cycle_loss(real_y, cycled_y)

        g_total = multiscale_gen_loss(discriminator_y, real_y, fake_y) + cyc + identity_loss(real_y, same_y)
        f_total = multiscale_gen_loss(discriminator_x, real_x, fake_x) + cyc + identity_loss(real_x, same_x)
        dx_loss = multiscale_disc_loss(discriminator_x, real_x, fake_x_buf)
        dy_loss = multiscale_disc_loss(discriminator_y, real_y, fake_y_buf)

    g_g_opt.apply_gradients(zip(tape.gradient(g_total, generator_g.trainable_variables), generator_g.trainable_variables))
    g_f_opt.apply_gradients(zip(tape.gradient(f_total, generator_f.trainable_variables), generator_f.trainable_variables))
    d_x_opt.apply_gradients(zip(tape.gradient(dx_loss, discriminator_x.trainable_variables), discriminator_x.trainable_variables))
    d_y_opt.apply_gradients(zip(tape.gradient(dy_loss, discriminator_y.trainable_variables), discriminator_y.trainable_variables))
    del tape
    return g_total, f_total, dx_loss, dy_loss

@tf.function
def distributed_train_step(dist_inputs):
    def step_fn(inputs):
        real_x, real_y = inputs
        return train_step(real_x, real_y)

    g_total, f_total, dx_loss, dy_loss = strategy.run(step_fn, args=(dist_inputs,))
    g_total = strategy.reduce(tf.distribute.ReduceOp.MEAN, g_total, axis=None)
    f_total = strategy.reduce(tf.distribute.ReduceOp.MEAN, f_total, axis=None)
    dx_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, dx_loss, axis=None)
    dy_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, dy_loss, axis=None)
    return g_total, f_total, dx_loss, dy_loss

for batch in train_ds.take(1):
    print(distributed_train_step(batch))
    break

STEPS_PER_EPOCH = 400

for epoch in range(1, EPOCHS+1):
    update_lr(epoch)
    print(f"\nEpoch {epoch}/{EPOCHS}  lr={g_g_opt.learning_rate.numpy():.2e}")
    for step, batch in enumerate(train_ds.take(STEPS_PER_EPOCH), 1):
        g1, g2, d1, d2 = distributed_train_step(batch)
        if step % 100 == 0:
            print(f"Step {step}: Gg={g1:.3f} Gf={g2:.3f} Dx={d1:.3f} Dy={d2:.3f}")

print("\nTraining done!")

# generation
def tensor_to_img(t):
    t = (t + 1)*127.5
    t = tf.clip_by_value(t, 0, 255)
    return tf.cast(t, tf.uint8).numpy()

def save_img(t, path):
    img = tf.keras.preprocessing.image.array_to_img(tensor_to_img(t))
    img.save(path, "JPEG")

print("\nGenerating Monet images...")
gen_ds = load_dataset(PHOTO_TFREC_PATTERN, 1, augment=False)

n = 0
for batch in gen_ds:
    fake = generator_g(batch, training=False)
    for img in fake:
        if n >= NUM_GENERATED_IMAGES: break
        save_img(img, f"{OUTPUT_DIR}/image_{n:05d}.jpg")
        n += 1
    if n >= NUM_GENERATED_IMAGES: break

print("Saved:", n)

# zip
zip_path = "/kaggle/working/images.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith(".jpg"):
            z.write(os.path.join(OUTPUT_DIR, f), f)

print("Zipped:", zip_path)

2026-03-16 14:45:19.429296: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773672319.602867      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773672319.653519      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773672320.070737      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773672320.070777      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773672320.070780      24 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0


I0000 00:00:1773672343.755583      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1773672396.768368      65 cuda_dnn.cc:529] Loaded cuDNN version 91002


(<tf.Tensor: shape=(), dtype=float32, numpy=40.04475021362305>, <tf.Tensor: shape=(), dtype=float32, numpy=38.12845993041992>, <tf.Tensor: shape=(), dtype=float32, numpy=1.567125916481018>, <tf.Tensor: shape=(), dtype=float32, numpy=1.921669602394104>)

Epoch 1/40  lr=2.00e-04
Step 100: Gg=28.217 Gf=27.807 Dx=0.168 Dy=0.268
Step 200: Gg=28.033 Gf=28.820 Dx=0.211 Dy=0.152
Step 300: Gg=28.193 Gf=27.395 Dx=0.053 Dy=0.106
Step 400: Gg=29.128 Gf=29.289 Dx=0.046 Dy=0.067

Epoch 2/40  lr=2.00e-04
Step 100: Gg=31.644 Gf=28.784 Dx=0.355 Dy=0.149
Step 200: Gg=26.186 Gf=25.890 Dx=0.257 Dy=0.302
Step 300: Gg=32.082 Gf=28.287 Dx=0.057 Dy=0.080
Step 400: Gg=26.587 Gf=28.395 Dx=0.092 Dy=0.117

Epoch 3/40  lr=2.00e-04
Step 100: Gg=28.355 Gf=27.154 Dx=0.067 Dy=0.108
Step 200: Gg=26.436 Gf=26.794 Dx=0.245 Dy=0.127
Step 300: Gg=25.264 Gf=28.292 Dx=0.151 Dy=0.149
Step 400: Gg=25.774 Gf=26.731 Dx=0.129 Dy=0.125

Epoch 4/40  lr=2.00e-04
Step 100: Gg=23.604 Gf=27.701 Dx=0.556 Dy=0.146
Step 200: Gg=25.955 Gf=